In [1]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

# Policy inference

The following example shows how to create a policy from a checkpoint and run inference on a dummy example.

In [2]:
config = _config.get_config("pi0_fast_droid")
checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_fast_droid")


In [3]:
# Check cache directory and any partial downloads
import pathlib
from openpi.shared import download

cache_dir = download.get_cache_dir()
print(f"Cache directory: {cache_dir}")
print(f"Cache directory exists: {cache_dir.exists()}")

if cache_dir.exists():
    print(f"Cache contents:")
    for item in cache_dir.iterdir():
        if item.is_dir():
            print(f"  - {item.name}/ (directory)")
        else:
            print(f"  - {item.name}")

# Check for any partial downloads (files ending with .partial)
partial_files = list(cache_dir.rglob("*.partial"))
if partial_files:
    print(f"\nFound {len(partial_files)} partial download(s):")
    for partial_file in partial_files:
        print(f"  - {partial_file}")
else:
    print("\nNo partial downloads found")


Cache directory: /home/skr/.cache/openpi
Cache directory exists: True
Cache contents:
  - big_vision/ (directory)
  - openpi-assets/ (directory)

Found 1 partial download(s):
  - /home/skr/.cache/openpi/openpi-assets/checkpoints/pi0_aloha_sim.partial


In [4]:

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
result = policy.infer(example)

# Delete the policy to free up memory.
del policy

print("Actions shape:", result["actions"].shape)

Actions shape: (10, 8)


In [ ]:
# Don't run this cell as it take too much memory
#  Use a smaller model configuration to reduce memory usage
config = _config.get_config("pi0_fast_droid")  # Use the smaller model instead

checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_fast_droid")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory immediately
del model

print("Loss shape:", loss.shape)


2025-10-17 16:13:43.796000: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.50GiB (rounded to 4831838208)requested by op 
2025-10-17 16:13:43.796085: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ***************************************************_________________________________________________


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 4831838208 bytes.

: 

# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [ ]:
# Don't run this cell as it take too much memory
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

2025-10-17 16:05:33.019795: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.25GiB (rounded to 2415919104)requested by op 
2025-10-17 16:05:33.019970: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] *******************************************************************************************_________
2025-10-17 16:05:43.247356: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.96GiB (rounded to 2106589184)requested by op 
2025-10-17 16:05:43.247453: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] *******************************************************************************************_________


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 2415919104 bytes.

: 

Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [5]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)

ValueError: Repo ID is not set. Cannot create dataset.